# Baseline en test con LM Studio

Evalúa los modelos sin entrenar sobre todo `datos/test.jsonl`: 20 árboles en cada uno de los 8 niveles, con `INTENTOS` intentos por árbol. Los episodios ya guardados no se repiten. Al terminar un modelo, sigue con el siguiente de `MODELOS`.

**Antes de empezar:** abre LM Studio, inicia el servidor local y evita que el PC entre en suspensión. LM Studio carga cada modelo cuando le toca. Abre este notebook desde la carpeta `seleccion_arbol` y selecciona el kernel del entorno de conda `tipico`, que tiene `pandas` y `numpy`.

**Si se interrumpe:** ejecuta todo de nuevo. Cada episodio se guarda al terminar, así que continúa donde quedó. No cambies el `nombre` de un modelo con episodios guardados, o empezaría de cero en otra carpeta.

**Intentos 4 a 6:** DeepSeek, Llama y OLMo ya tienen una segunda corrida en los niveles 2 a 6, guardada como intentos 4 a 6. `pass@k` usa todos los intentos de cada árbol.

**Ministral:** sus intentos 1 a 3 de los niveles 2 a 6 se corrieron con Transformers en 4 bits NF4. El resto de su test se completa aquí, en LM Studio con el GGUF Q4_K_M de `lmstudio-community`.

**Árboles extra:** al final, un bucle agrega un árbol nuevo por nivel a `datos/test_extra.jsonl` y evalúa todos los modelos en él, hasta que detengas la celda.

**Si cambias un presupuesto:** usa otra carpeta de resultados, para no mezclar corridas.

## Configuración

In [1]:
from evaluacion import leer_resultados
from modelo import DEEPSEEK, LLAMA, MINISTRAL, OLMO, QWEN, ModeloLMStudio

URL = 'http://localhost:1235/v1'
RUTA_DATOS = 'datos/test.jsonl'
NIVELES = sorted({ejemplo['nivel'] for ejemplo in leer_resultados(RUTA_DATOS)})  # Todos los niveles del test
INTENTOS = 4                                    # Intentos por árbol, para pass@4
TOKENS_PENSAMIENTO = 3400                       # Al agotarlos, se cierra el razonamiento y el modelo debe responder
TOKENS_RESPUESTA = 200                          # Tokens para escribir los nodos después de ese cierre

# Se evalúan en este orden. Un formato incorrecto no da error, pero altera los resultados.
# El nombre es la columna `modelo` y la carpeta de resultados: debe coincidir con la que ya existe.
# recargar_cada: recarga el modelo en LM Studio cada esos episodios, porque llama-server acumula en RAM
# la caché de los prompts anteriores hasta quedarse sin memoria.
MODELOS = [
    dict(nombre='DeepSeek R1 Distill 1.5B', lm_studio='deepseek-r1-distill-qwen-1.5b', formato=DEEPSEEK),
    dict(nombre='Qwen 3.5 2B', lm_studio='qwen3.5-2b', formato=QWEN),
    dict(nombre='OLMo 2 1B', lm_studio='olmo-2-0425-1b-instruct', formato=OLMO),
    dict(nombre='Llama 3.2 1B', lm_studio='llama-3.2-1b-instruct', formato=LLAMA),
    # Solo completa el test: no se evalúa en los árboles extra.
    dict(nombre='Ministral 3 3B Reasoning 4bit', lm_studio='ministral-3-3b-reasoning-2512', formato=MINISTRAL, extra=False, recargar_cada=3),
]
RUTAS = [f"resultados/{m['nombre']}/baseline_test.jsonl" for m in MODELOS]

## Comprobar la conexión

Si esta celda falla, revisa que el servidor de LM Studio esté iniciado.

In [2]:
import json
import urllib.request

with urllib.request.urlopen(f'{URL}/models', timeout=10) as respuesta:
    disponibles = [m['id'] for m in json.load(respuesta)['data']]

print('Modelos disponibles:', disponibles)
faltantes = [m['lm_studio'] for m in MODELOS if m['lm_studio'] not in disponibles]
assert not faltantes, f'No están disponibles en LM Studio: {faltantes}'

Modelos disponibles: ['ministral-3-3b-reasoning-2512', 'deepseek-r1-distill-qwen-1.5b', 'olmo-2-0425-1b-instruct', 'llama-3.2-1b-instruct', 'qwen3.5-2b', 'text-embedding-nomic-embed-text-v1.5']


## Datos de test

Resumen por nivel y todos los árboles del test, incluidos los niveles 1, 7 y 8. La tabla completa omite `prompt` y `razonamiento`, que son textos largos.

- **atajo_optimo:** árboles donde la estrategia "sin vecinos" ya alcanza el óptimo.

In [3]:
import pandas as pd

test = pd.DataFrame(leer_resultados(RUTA_DATOS))
test['atajo_optimo'] = test['puntaje_sin_vecinos'] == test['puntaje_optimo']
test.groupby('nivel').agg(
    arboles=('id', 'size'),
    nodos=('nodos', 'first'),
    costo_medio=('costo', 'mean'),
    optimo_medio=('puntaje_optimo', 'mean'),
    voraz_medio=('puntaje_voraz', 'mean'),
    sin_vecinos_medio=('puntaje_sin_vecinos', 'mean'),
    atajo_optimo=('atajo_optimo', 'mean'),
).round(2)

,arboles,nodos,costo_medio,optimo_medio,voraz_medio,sin_vecinos_medio,atajo_optimo
nivel,,,,,,,
1,20,3,2.35,10.25,5.1,9.65,0.85
2,20,4,2.30,14.20,6.1,13.55,0.70
3,20,5,2.60,14.45,6.8,10.90,0.00
4,20,7,3.05,22.20,9.7,16.70,0.00
5,20,10,3.90,35.55,14.7,28.35,0.00
6,20,15,4.55,78.05,32.9,66.65,0.00
7,20,25,6.15,184.00,105.9,156.65,0.00
8,20,40,8.35,385.30,220.3,334.85,0.00


In [4]:
numero = test['id'].str.rsplit('-', n=1).str[1].astype(int)
tabla_test = test.assign(numero=numero).sort_values(['nivel', 'numero']).drop(columns=['numero', 'prompt', 'razonamiento'])

with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.max_colwidth', None):
    display(tabla_test.reset_index(drop=True))

,id,nivel,nodos,valores,aristas,costo,respuesta,puntaje_optimo,puntaje_voraz,puntaje_sin_vecinos,atajo_optimo
0,test-1-0,1,3,"[4, 1, 1]","[[1, 2], [3, 1]]",1,<respuesta>1</respuesta>,4,2,4,True
1,test-1-1,1,3,"[6, 5, 3]","[[1, 2], [2, 3]]",2,<respuesta>1 3</respuesta>,9,6,9,True
2,test-1-2,1,3,"[2, 10, 2]","[[3, 1], [2, 1]]",2,<respuesta>2 3</respuesta>,12,6,12,True
3,test-1-3,1,3,"[2, 1, 3]","[[2, 3], [2, 1]]",2,<respuesta>1 3</respuesta>,5,0,5,True
4,test-1-4,1,3,"[8, 9, 2]","[[1, 2], [3, 1]]",3,<respuesta>2 3</respuesta>,11,7,11,True
5,test-1-5,1,3,"[5, 5, 5]","[[3, 1], [1, 2]]",3,<respuesta>2 3</respuesta>,10,3,5,False
6,test-1-6,1,3,"[2, 2, 1]","[[2, 3], [2, 1]]",2,<respuesta>1 3</respuesta>,3,0,3,True
7,test-1-7,1,3,"[1, 8, 1]","[[2, 3], [1, 2]]",2,<respuesta>2</respuesta>,8,2,8,True
8,test-1-8,1,3,"[8, 3, 4]","[[3, 2], [1, 2]]",3,<respuesta>1 3</respuesta>,12,3,12,True
9,test-1-9,1,3,"[6, 8, 7]","[[1, 3], [1, 2]]",3,<respuesta>2 3</respuesta>,15,9,15,True


## Evaluación

Esta celda sigue hasta terminar todos los modelos. Si un modelo falla, pasa al siguiente y lo reintenta al final, hasta 5 rondas con 5 minutos de pausa.

Los modelos con `recargar_cada` se recargan con `lms`, el CLI de LM Studio, todo en GPU y con contexto de `CONTEXTO`. Cada recarga tarda unos 15 s.

Para ver los resultados antes de que termine, interrumpe la celda y ejecuta las de abajo. Al volver a ejecutarla, continúa donde quedó y solo se repite el episodio en curso.

In [5]:
import shutil
import subprocess
from pathlib import Path

from evaluacion import evaluar_varios
from modelo import ModeloLLM

LMS = shutil.which('lms') or str(Path.home() / '.lmstudio' / 'bin' / 'lms.exe')
CONTEXTO = 8192  # Alcanza para el prompt, el pensamiento y la respuesta


class ConRecarga(ModeloLLM):
    # Recarga el modelo antes de la primera consulta y después cada `cada` consultas.
    def __init__(self, modelo, cada):
        self.modelo = modelo
        self.cada = cada
        self.consultas = 0

    def recargar(self):
        subprocess.run([LMS, 'unload', self.modelo.nombre], capture_output=True)
        carga = subprocess.run(
            [LMS, 'load', self.modelo.nombre, '--gpu', 'max', '--context-length', str(CONTEXTO), '-y'],
            capture_output=True, encoding='utf-8', errors='replace',
        )
        if carga.returncode != 0:
            # RuntimeError: el episodio no se guarda y se reintenta, como cualquier falla de LM Studio.
            raise RuntimeError(f'No se pudo recargar {self.modelo.nombre}: {carga.stderr[-300:]}')

    def generar(self, prompt):
        if self.consultas % self.cada == 0:
            self.recargar()
        self.consultas += 1
        return self.modelo.generar(prompt)

    def __getattr__(self, nombre):
        # La evaluación lee ultimo_tokens, ultimo_fin y los demás datos de la última consulta.
        return getattr(self.modelo, nombre)


def crear_modelo(m):
    modelo = ModeloLMStudio(
        nombre=m['lm_studio'],
        formato=m['formato'],
        url_base=URL,
        tokens_pensamiento=TOKENS_PENSAMIENTO,
        tokens_respuesta=TOKENS_RESPUESTA,
    )
    return ConRecarga(modelo, m['recargar_cada']) if 'recargar_cada' in m else modelo


evaluaciones = [
    (crear_modelo(m), m['nombre'], ruta)
    for m, ruta in zip(MODELOS, RUTAS)
]

incompletos = evaluar_varios(evaluaciones, RUTA_DATOS, niveles=NIVELES, intentos=INTENTOS)
print('Modelos con episodios pendientes:', incompletos or 'ninguno')

Ministral 3 3B Reasoning 4bit nivel 7 ep test-7-10 intento 4: recompensa=0.000 puntaje=-99 óptimo=190 126.0s 3464tok
Ministral 3 3B Reasoning 4bit nivel 7 ep test-7-16 intento 1: recompensa=0.106 puntaje=21 óptimo=111 115.8s 3405tok
Ministral 3 3B Reasoning 4bit nivel 7 ep test-7-16 intento 2: recompensa=0.030 puntaje=6 óptimo=111 115.0s 3411tok
Ministral 3 3B Reasoning 4bit nivel 7 ep test-7-16 intento 3: recompensa=0.000 puntaje=-136 óptimo=111 126.5s 3445tok
Ministral 3 3B Reasoning 4bit nivel 7 ep test-7-16 intento 4: recompensa=0.141 puntaje=28 óptimo=111 114.0s 3407tok
Ministral 3 3B Reasoning 4bit nivel 8 ep test-8-7 intento 1: recompensa=0.022 puntaje=13 óptimo=340 117.0s 3406tok
Ministral 3 3B Reasoning 4bit nivel 8 ep test-8-7 intento 2: recompensa=0.068 puntaje=40 óptimo=340 126.9s 3406tok
Ministral 3 3B Reasoning 4bit nivel 8 ep test-8-7 intento 3: recompensa=0.075 puntaje=44 óptimo=340 115.0s 3404tok
Ministral 3 3B Reasoning 4bit nivel 8 ep test-8-7 intento 4: recompensa=0

## Baseline por nivel

Aparecen todos los modelos y niveles: los que aún no se evalúan quedan vacíos. No ejecutes la celda de evaluación en dos kernels a la vez: se repetirían episodios.

Las columnas de proporción van de 0 a 1.

- **pass@1:** intentos que logran el óptimo.
- **pass@4:** probabilidad de que 4 intentos de un árbol incluyan al menos un óptimo.
- **recompensa:** media del episodio. Igualar el mejor atajo vale 0,5, y una respuesta inválida, -1.
- **optimos:** árboles donde el modelo logró el mejor puntaje posible.
- **optimos_atajo:** óptimos de la estrategia "sin vecinos", que no razona sobre costos. El modelo debería superarla para mostrar que optimiza de verdad.
- **supera_voraz:** veces que superó la regla simple.
- **supera_atajos:** veces que superó al mejor atajo.
- **invalidos:** respuestas sin un formato válido.
- **cortes:** episodios que agotaron el pensamiento, y cuya respuesta fue forzada.

In [6]:
import pandas as pd

from evaluacion import resumir, resumir_pass

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
todos = pd.MultiIndex.from_product([[m['nombre'] for m in MODELOS], NIVELES], names=['modelo', 'nivel'])

resumir_pass(RUTAS, k=INTENTOS).reindex(todos)

intentos  pass@1  arboles_completos  \
modelo                        nivel                                        
DeepSeek R1 Distill 1.5B      1            80   0.325                 20   
                              2           120   0.308                 20   
                              3           120   0.142                 20   
                              4           120   0.017                 20   
                              5           120   0.000                 20   
                              6           120   0.000                 20   
                              7            80   0.000                 20   
                              8            80   0.000                 20   
Qwen 3.5 2B                   1            80   0.288                 20   
                              2            80   0.175                 20   
                              3            80   0.012                 20   
                              4            80   0.000                 20   
                              5            80   0.000                 20   
                              6            80   0.000                 20   
                              7            80   0.000                 20   
                              8            80   0.000                 20   
OLMo 2 1B                     1            80   0.275                 20   
                              2           120   0.225                 20   
                              3           120   0.000                 20   
                              4           120   0.000                 20   
                              5           120   0.000                 20   
                              6           120   0.000                 20   
                              7            80   0.000                 20   
                              8            80   0.000                 20   
Llama 3.2 1B                  1            80   0.288                 20   
                              2           120   0.208                 20   
                              3           120   0.000                 20   
                              4           120   0.000                 20   
                              5           120   0.000                 20   
                              6           120   0.000                 20   
                              7            80   0.000                 20   
                              8            80   0.000                 20   
Ministral 3 3B Reasoning 4bit 1            80   0.875                 20   
                              2            80   0.675                 20   
                              3            80   0.500                 20   
                              4            80   0.188                 20   
                              5            80   0.100                 20   
                              6            80   0.000                 20   
                              7            80   0.000                 20   
                              8            80   0.000                 20   

                                     pass@4  
modelo                        nivel          
DeepSeek R1 Distill 1.5B      1       0.500  
                              2       0.637  
                              3       0.417  
                              4       0.067  
                              5       0.000  
                              6       0.000  
                              7       0.000  
                              8       0.000  
Qwen 3.5 2B                   1       0.500  
                              2       0.450  
                              3       0.050  
                              4       0.000  
                              5       0.000  
                              6       0.000  
                              7       0.000  
                              8       0.000  
OLMo 2 1B       

In [7]:
resumir(RUTAS).reindex(todos)

episodios  recompensa  optimos  \
modelo                        nivel                                   
DeepSeek R1 Distill 1.5B      1             80       0.551    0.325   
                              2            120       0.512    0.308   
                              3            120       0.447    0.142   
                              4            120       0.275    0.017   
                              5            120       0.156    0.000   
                              6            120       0.102    0.000   
                              7             80       0.035    0.000   
                              8             80       0.031    0.000   
Qwen 3.5 2B                   1             80       0.569    0.288   
                              2             80       0.378    0.175   
                              3             80       0.234    0.012   
                              4             80       0.146    0.000   
                              5             80       0.089    0.000   
                              6             80       0.048    0.000   
                              7             80      -0.020    0.000   
                              8             80      -0.021    0.000   
OLMo 2 1B                     1             80       0.396    0.275   
                              2            120       0.356    0.225   
                              3            120       0.026    0.000   
                              4            120      -0.091    0.000   
                              5            120      -0.222    0.000   
                              6            120      -0.198    0.000   
                              7             80      -0.286    0.000   
                              8             80      -0.271    0.000   
Llama 3.2 1B                  1             80       0.564    0.288   
                              2            120       0.510    0.208   
                              3            120       0.290    0.000   
                              4            120       0.193    0.000   
                              5            120       0.122    0.000   
                              6            120       0.111    0.000   
                              7             80       0.054    0.000   
                              8             80       0.040    0.000   
Ministral 3 3B Reasoning 4bit 1             80       0.810    0.875   
                              2             80       0.643    0.675   
                              3             80       0.588    0.500   
                              4             80       0.302    0.188   
                              5             80       0.233    0.100   
                              6             80      -0.097    0.000   
                              7             80       0.090    0.000   
                              8             80       0.057    0.000   

                                     optimos_atajo  supera_voraz  \
modelo                        nivel                                
DeepSeek R1 Distill 1.5B      1               0.85         0.600   
                              2               0.70         0.667   
                              3               0.00         0.608   
                              4               0.00         0.400   
                              5               0.00         0.275   
                              6               0.00         0.217   
                              7               0.00         0.050   
                              8               0.00         0.012   
Qwen 3.5 2B                   1               0.85         0.562   
                              2               0.70         0.438   
                              3               0.00         0.438   
                              4               0.00         0.262   
                              5               0.00         0.212   
                              6   

## Baseline total

Mientras un modelo no complete todos los niveles, su total no se compara con el de los demás, porque promedia niveles distintos.

In [8]:
resumir_pass(RUTAS, k=INTENTOS, por=['modelo'])

,intentos,pass@1,arboles_completos,pass@4
modelo,,,,
DeepSeek R1 Distill 1.5B,840,0.098,160,0.202
Llama 3.2 1B,840,0.057,160,0.096
Ministral 3 3B Reasoning 4bit,640,0.292,160,0.475
OLMo 2 1B,840,0.058,160,0.075
Qwen 3.5 2B,640,0.059,160,0.125


In [9]:
resumir(RUTAS, por=['modelo'])

,episodios,recompensa,optimos,optimos_atajo,supera_voraz,supera_atajos,invalidos,cortes,segundos,tokens
modelo,,,,,,,,,,
DeepSeek R1 Distill 1.5B,840,0.272,0.098,0.181,0.373,0.071,0.024,0.980,104.998,3408.237
Llama 3.2 1B,840,0.238,0.057,0.181,0.276,0.032,0.000,0.000,1.591,4.544
Ministral 3 3B Reasoning 4bit,640,0.328,0.292,0.194,0.528,0.191,0.123,0.717,135.277,3124.127
OLMo 2 1B,840,-0.034,0.058,0.181,0.240,0.020,0.238,0.000,2.975,17.998
Qwen 3.5 2B,640,0.178,0.059,0.194,0.255,0.017,0.048,1.000,88.481,3413.934


## Árboles extra en bucle

Cada ronda agrega a `datos/test_extra.jsonl` un árbol nuevo por nivel, distinto de todos los de `datos`. Después evalúa los modelos de `MODELOS`, salvo los marcados con `extra=False`, en ese árbol, con `INTENTOS` intentos cada uno. Los resultados van a `resultados/<modelo>/baseline_test_extra.jsonl`. Una ronda tarda entre 1 y 2 horas, sobre todo por DeepSeek y Qwen.

Sigue hasta que detengas la celda. Al volver a ejecutarla, primero termina los episodios pendientes y después agrega árboles. Requiere haber ejecutado la celda de evaluación, que define `evaluaciones`.

In [ ]:
from dataset import agregar_arboles

RUTA_EXTRA = 'datos/test_extra.jsonl'
CON_EXTRA = [(evaluacion, m) for evaluacion, m in zip(evaluaciones, MODELOS) if m.get('extra', True)]
RUTAS_EXTRA = [f"resultados/{m['nombre']}/baseline_test_extra.jsonl" for _, m in CON_EXTRA]
evaluaciones_extra = [(modelo, nombre, ruta) for ((modelo, nombre, _), _), ruta in zip(CON_EXTRA, RUTAS_EXTRA)]

while True:
    # Se evalúa antes de agregar, así al reanudar se terminan primero los árboles que quedaron pendientes.
    incompletos = evaluar_varios(evaluaciones_extra, RUTA_EXTRA, intentos=INTENTOS)
    if incompletos:
        print('Modelos con episodios pendientes, se reintentan:', incompletos)
        continue
    nuevos = agregar_arboles(RUTA_EXTRA)
    if not nuevos:
        print('No quedan árboles nuevos en ningún nivel')
        break
    print('Árboles agregados:', [e['id'] for e in nuevos])


Nivel 1: no se encontró un árbol nuevo en 1000 intentos
Árboles agregados: ['test_extra-2-17', 'test_extra-3-17', 'test_extra-4-17', 'test_extra-5-17', 'test_extra-6-17', 'test_extra-7-17', 'test_extra-8-17']
DeepSeek R1 Distill 1.5B nivel 2 ep test_extra-2-17 intento 1: recompensa=0.556 puntaje=5 óptimo=9 98.3s 3404tok
DeepSeek R1 Distill 1.5B nivel 2 ep test_extra-2-17 intento 2: recompensa=0.333 puntaje=3 óptimo=9 82.4s 3404tok
DeepSeek R1 Distill 1.5B nivel 2 ep test_extra-2-17 intento 3: recompensa=0.667 puntaje=6 óptimo=9 84.2s 3404tok
DeepSeek R1 Distill 1.5B nivel 2 ep test_extra-2-17 intento 4: recompensa=0.667 puntaje=6 óptimo=9 83.8s 3404tok
DeepSeek R1 Distill 1.5B nivel 3 ep test_extra-3-17 intento 1: recompensa=0.438 puntaje=14 óptimo=21 82.6s 3404tok
DeepSeek R1 Distill 1.5B nivel 3 ep test_extra-3-17 intento 2: recompensa=0.500 puntaje=16 óptimo=21 82.4s 3404tok
DeepSeek R1 Distill 1.5B nivel 3 ep test_extra-3-17 intento 3: recompensa=0.438 puntaje=14 óptimo=21 82.0s 34

## Baseline con los árboles extra

Junta el test y los árboles extra ya evaluados. Ejecútala después de detener el bucle. Un árbol entra en `pass@4` solo si tiene sus 4 intentos.

In [ ]:
resumir_pass(RUTAS + RUTAS_EXTRA, k=INTENTOS).reindex(todos)


In [ ]:
resumir_pass(RUTAS + RUTAS_EXTRA, k=INTENTOS, por=['modelo'])
